# WWTD-2025 (What Would Trump Do?)

Generate a forecasting dataset about Trump's actions, decisions, and statements using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments—including evaluation with and without context.

In [1]:
%pip install lightningrod-ai python-dotenv pandas openai

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [2]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for Trump-related forecasting.

In [3]:
instructions = """
Generate binary forecasting questions about Trump's actions, decisions, positions, and statements.
Questions should be diverse, related to the content, and should evenly cover the full range from very likely to very unlikely.
Horizon: outcomes should be known within 2 months of the question date, and may be known much sooner.
Criteria: binary outcome, exact dates, self-contained, verifiable via web search, newsworthy.
"""

good_examples = [
    "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2025?",
    "Will Trump issue pardons to January 6 defendants within his first week in office?",
    "Will Pete Hegseth be confirmed as Secretary of Defense by February 15, 2025?",
    "Will Trump sign an executive order to keep TikTok operational in the US by January 31, 2025?",
    "Will Kash Patel be confirmed as FBI Director by March 1, 2025?",
]

bad_examples = [
    "Will Trump do something controversial? (too vague)",
    "Will Trump be in the news? (obvious)",
    "Will tariffs be imposed? (needs specifics)",
]

In [4]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2025, 1, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=7,
        search_query=[
            "Donald Trump domestic policy agenda",
            "Donald Trump trade and tariff actions",
            "Donald Trump foreign policy decisions",
            "Donald Trump interviews and press appearances",
            "Donald Trump lawsuits and court rulings",
        ],
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=1,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [5]:
dataset = lr.transforms.run(pipeline, max_questions=20, name="WWTD-2025")

samples = dataset.download()
pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $0.03                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃ In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons   ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ NewsSeedGenerator… │ Complete             │  1 │  10 │        0 │      0 │ -                   │       2s │  │
│  │ ForwardLookingQue… │ Complete             │ 10 │  50 │        0 │      0 │ -                   │       1s │  │
│  │ WebSearchLabelerT… │ Complete             │ 50 │  45 │        5 │      0 │ Resolution date is  │       1s │  │
│  │                    │                      │    │     │          │        │ before seed         │          │  │
│  │                    │                      │    │     │          │        │ creation date (4),  │          │  │
│  │                    │                      │    │     │          │        │ Low confidence:     │          │  │
│  │                    │                      │    │     │          │        │ 0.80 < 0.9 (1)      │          │  │
│  │ NewsContextGenera… │ Complete             │ 45 │  45 │        0 │      0 │ -                   │       1s │  │
│  └────────────────────┴──────────────────────┴────┴─────┴──────────┴────────┴─────────────────────┴──────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

50 samples (90.0% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [ ]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(1, 60)),
    split=SplitParams(test_size=0.2),
)

for name, ds in [("Train", train_dataset), ("Test", test_dataset)]:
    data = ds.flattened()
    print(len(data))
    yes_count = sum(1 for s in data if s.get("label") in (1, "1", 1.0))
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    display(pd.DataFrame(data).head())

[prepare_for_training] Starting with 50 samples
[filter] Dropped 5 invalid, 16 horizon → 29 remain
[dedup] 29 remain (0 duplicates)
[split] 23 train samples removed for leakage
[split] Temporal split: 0 train, 6 test


ValueError: [filter_and_split] Unhealthy split detected.

23/23 train samples (100%) were removed for temporal leakage — the date_close or resolution_date of train questions extends into the test period.

Tips:
  - Use test_start="YYYY-MM-DD" instead of test_size to set an explicit cutoff at least 60 days before your last question date, giving train questions room to resolve before the test window.
  - Tighten days_to_resolution_range — the current max of 60 days means train resolution dates extend far into the test window. Reducing it shrinks the bleed-over zone.
  - Generate more samples across a wider date range. With questions spread over a longer period, the temporal split cutoff moves far enough back that earlier questions resolve well before the test window.
  - Set filter_leaky_train=False to disable leakage removal. Only do this if you are confident the resolution dates do not reveal information that was unavailable at prediction time.

## Model Training

Fine-tune a forecasting model on your dataset. For production training, generate more questions (increase `max_questions` or run without limit). Our reference experiments used 2,790 questions—see [Trump-Forecaster Model](https://huggingface.co/LightningRodLabs/Trump-Forecaster) and [Trump-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025) for details.

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [ ]:
from lightningrod import TrainingConfig

config = TrainingConfig(
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.32
Effective steps: 11
Train tokens: 1,073,959
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.

In [ ]:
job = lr.training.run(config, dataset=train_dataset, name="WWTD-2025")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: WWTD-2025                                                                                               │
│                                                                                                                 │
│    Reward: latest -0.8786  avg -0.6684  (11 steps)  (higher is better)                                          │
│                                                                                                                 │
│    Cost:  $0.18                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 13fa02ec-27f4-47a9-84c9-762d91a1904a completed with status: COMPLETED
Trained model ID: checkpoint:13fa02ec-27f4-47a9-84c9-762d91a1904a


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model.

In [ ]:
print(lr.predict(job.model_id, "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2027?"))

<answer>0.05</answer>


## Run evals on trained model

Run test evals on your trained model against the test dataset. The eval job runs the model on the dataset and reports metrics.

In [ ]:
eval_job = lr.evals.run(model_id=job.model_id, dataset=test_dataset, benchmark_model_id="openai/gpt-5.2")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: 00602447-5872-4732-93f9-b0d99459da1a                                                                     │
│    Model: checkpoint:13fa02ec-27f4-47a9-84c9-762d91a1904a                                                       │
│    Dataset: 82186c26-a309-43a6-9543-37bdda38d41d                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┓                                                        │
│  ┃ Metric              ┃    base ┃ trained ┃ benchmark ┃                                                        │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━┩                                                        │
│  │ brier_score         │  0.2333 │  0.1877 │    0.1555 │                                                        │
│  │ ece                 │  0.1451 │  0.0963 │    0.0590 │                                                        │
│  │ mean_reward         │ -0.7840 │ -0.6048 │   -0.4966 │                                                        │
│  │ mean_valid_reward   │ -0.7840 │ -0.6048 │   -0.4966 │                                                        │
│  │ n_samples           │     113 │     113 │       113 │                                                        │
│  │ n_valid             │     113 │     113 │       113 │                                                        │
│  │ parse_rate          │  1.0000 │  1.0000 │    1.0000 │                                                        │
│  │ total_cost          │  0.0068 │  0.0068 │         — │                                                        │
│  │ total_input_tokens  │   93344 │   93344 │     88060 │                                                        │
│  │ total_output_tokens │    1111 │    1101 │     28947 │                                                        │
│  └─────────────────────┴─────────┴─────────┴───────────┘                                                        │
│                                                                                                                 │
│    Cost:  $0.01                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Note: the trained model checkpoint will only be available for 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.